# Compare original ALBEF vs FC-fused multi-view heatmaps

This notebook compares:

- **original single-view ALBEF ITC-margin Grad-CAM heatmaps**, and
- **new FC-fused multi-view ITC-margin Grad-CAM heatmaps**

for the same VinDr-CXR image-label cases.

It produces a final **multi-page PDF** in which each row shows:

1. original image + GT box
2. original ALBEF heatmap
3. original ALBEF overlay
4. FC-fused heatmap
5. FC-fused overlay

Optionally, the notebook can also generate a **branch-detail PDF** showing:

- original branch overlay
- lung branch overlay
- heart branch overlay
- final fused overlay

Edit the path cell below before running.

In [ ]:
from pathlib import Path
import math
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

TARGET_LABELS = ["Cardiomegaly", "Pleural Effusion"]
EXPECTED_IMAGE_RES = 256
CASES_PER_PAGE = 4
OVERLAY_ALPHA = 0.50
CMAP_NAME = 'magma'
FIG_DPI = 150
SAVE_OUTPUTS = True
GENERATE_BRANCH_DETAIL_PDF = True

In [ ]:
# ------------------------------------------------------------------
# Edit these paths
# ------------------------------------------------------------------
ORIGINAL_ALBEF_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/original_albef_itc_margin_gradcam_test_best_cardio'
)
FUSED_FC_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/fc_fusion_itc_margin_gradcam_test_best_cardio'
)

IMAGES_ROOT = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test')
ANNOTATIONS_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/annotations_test.csv')
IMAGE_METADATA_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv')

VIS_OUTPUT_DIR = Path('/home/vault/iwi5/iwi5362h/results/visualization/original_vs_fc_fusion_heatmaps')
VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_INDEX_CSV = ORIGINAL_ALBEF_DIR / 'itc_margin_gradcam_index.csv'
FUSED_INDEX_CSV = FUSED_FC_DIR / 'fc_fusion_itc_margin_gradcam_index.csv'

for path in [ORIGINAL_ALBEF_DIR, FUSED_FC_DIR, IMAGES_ROOT, ANNOTATIONS_CSV, IMAGE_METADATA_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)

print('Original ALBEF dir :', ORIGINAL_ALBEF_DIR)
print('FC fused dir       :', FUSED_FC_DIR)
print('Visualization dir  :', VIS_OUTPUT_DIR)
print('Original index CSV :', ORIGINAL_INDEX_CSV, '| exists =', ORIGINAL_INDEX_CSV.exists())
print('FC fused index CSV :', FUSED_INDEX_CSV, '| exists =', FUSED_INDEX_CSV.exists())

## Helper functions

In [ ]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def as_2d_numpy(value):
    if torch.is_tensor(value):
        value = value.detach().cpu().float().numpy()
    return np.asarray(value, dtype=np.float32).squeeze()


def choose_column(df, candidates, purpose):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    raise KeyError(f'Cannot find {purpose}. Available columns: {list(df.columns)}')


def resolve_label_key(payload, requested_label):
    lookup = {
        str(key).casefold(): key
        for key in payload.keys()
        if str(key) != '__metadata__'
    }
    key = lookup.get(str(requested_label).casefold())
    if key is None:
        available = [k for k in payload.keys() if str(k) != '__metadata__']
        raise KeyError(f'Label {requested_label!r} not present. Available: {available}')
    return key


def find_payload_path(base_dir, image_id):
    direct = base_dir / f'{image_id}.pt'
    if direct.is_file():
        return direct
    maps_dir = base_dir / 'maps' / f'{image_id}.pt'
    if maps_dir.is_file():
        return maps_dir
    matches = list(base_dir.rglob(f'{image_id}.pt'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not find {image_id}.pt under {base_dir}')


def build_payload_index(base_dir, preferred_index_csv=None):
    if preferred_index_csv is not None and Path(preferred_index_csv).is_file():
        df = pd.read_csv(preferred_index_csv)
        if 'image_id' not in df.columns:
            raise KeyError(f'{preferred_index_csv} has no image_id column')
        df['image_id'] = df['image_id'].astype(str)
        if 'heatmap_path' in df.columns:
            def resolve_saved_path(p):
                p = Path(str(p))
                if p.is_file():
                    return p
                fallback = base_dir / p.name
                if fallback.is_file():
                    return fallback
                fallback2 = base_dir / 'maps' / p.name
                if fallback2.is_file():
                    return fallback2
                return find_payload_path(base_dir, p.stem)
            df['payload_path'] = df['heatmap_path'].map(resolve_saved_path)
        else:
            df['payload_path'] = df['image_id'].map(lambda x: find_payload_path(base_dir, x))
        return df[['image_id', 'payload_path']].drop_duplicates('image_id').reset_index(drop=True)

    payloads = sorted(base_dir.rglob('*.pt'))
    if not payloads:
        raise FileNotFoundError(f'No .pt payloads found under {base_dir}')
    rows = []
    for path in payloads:
        rows.append({'image_id': path.stem, 'payload_path': path})
    return pd.DataFrame(rows).drop_duplicates('image_id').reset_index(drop=True)


def load_image(image_id):
    with Image.open(IMAGES_ROOT / f'{image_id}.png') as handle:
        return handle.convert('RGB')


def make_overlay(image, vis_map, alpha=OVERLAY_ALPHA):
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    heatmap = as_2d_numpy(vis_map)
    if heatmap.shape != rgb.shape[:2]:
        raise ValueError(f'Image/map mismatch: image={rgb.shape[:2]}, map={heatmap.shape}')
    color = colormaps[CMAP_NAME](np.clip(heatmap, 0, 1))[..., :3]
    alpha_map = (alpha * np.clip(heatmap, 0, 1))[..., None]
    return np.clip((1 - alpha_map) * rgb + alpha_map * color, 0, 1)

## Load annotations and metadata for GT boxes

In [ ]:
annotations_df = pd.read_csv(ANNOTATIONS_CSV)
metadata_df = pd.read_csv(IMAGE_METADATA_CSV)

ann_image_col = choose_column(annotations_df, ['image_id', 'imageid', 'image_name'], 'annotation image id')
ann_label_col = choose_column(annotations_df, ['class_name', 'label', 'finding_name'], 'annotation label')
ann_xmin_col = choose_column(annotations_df, ['x_min', 'xmin', 'x1'], 'annotation x_min')
ann_ymin_col = choose_column(annotations_df, ['y_min', 'ymin', 'y1'], 'annotation y_min')
ann_xmax_col = choose_column(annotations_df, ['x_max', 'xmax', 'x2'], 'annotation x_max')
ann_ymax_col = choose_column(annotations_df, ['y_max', 'ymax', 'y2'], 'annotation y_max')

meta_image_col = choose_column(metadata_df, ['image_id', 'imageid', 'image_name'], 'metadata image id')
meta_width_col = choose_column(metadata_df, ['width', 'original_width', 'w'], 'metadata width')
meta_height_col = choose_column(metadata_df, ['height', 'original_height', 'h'], 'metadata height')

annotations_df = annotations_df.copy()
annotations_df[ann_image_col] = annotations_df[ann_image_col].astype(str)
annotations_df[ann_label_col] = annotations_df[ann_label_col].astype(str)
metadata_df = metadata_df.copy()
metadata_df[meta_image_col] = metadata_df[meta_image_col].astype(str)

metadata_lookup = {
    str(row[meta_image_col]): {
        'width': float(row[meta_width_col]),
        'height': float(row[meta_height_col]),
    }
    for _, row in metadata_df.iterrows()
}


def get_boxes(image_id, label, displayed_size):
    subset = annotations_df[
        (annotations_df[ann_image_col] == str(image_id))
        & (annotations_df[ann_label_col].str.casefold() == str(label).casefold())
    ]
    if subset.empty:
        return []
    if str(image_id) not in metadata_lookup:
        return []
    original = metadata_lookup[str(image_id)]
    sx = displayed_size[0] / original['width']
    sy = displayed_size[1] / original['height']
    boxes = []
    for _, row in subset.iterrows():
        boxes.append({
            'x': float(row[ann_xmin_col]) * sx,
            'y': float(row[ann_ymin_col]) * sy,
            'w': (float(row[ann_xmax_col]) - float(row[ann_xmin_col])) * sx,
            'h': (float(row[ann_ymax_col]) - float(row[ann_ymin_col])) * sy,
        })
    return boxes


def draw_boxes(ax, boxes, color='lime', lw=1.6):
    for box in boxes:
        ax.add_patch(Rectangle(
            (box['x'], box['y']), box['w'], box['h'],
            fill=False, edgecolor=color, linewidth=lw,
        ))

## Build the list of common comparable cases

In [ ]:
orig_index_df = build_payload_index(ORIGINAL_ALBEF_DIR, ORIGINAL_INDEX_CSV)
fused_index_df = build_payload_index(FUSED_FC_DIR, FUSED_INDEX_CSV)

orig_path_lookup = dict(zip(orig_index_df['image_id'], orig_index_df['payload_path']))
fused_path_lookup = dict(zip(fused_index_df['image_id'], fused_index_df['payload_path']))
common_image_ids = sorted(set(orig_path_lookup) & set(fused_path_lookup))

print('Original payloads :', len(orig_index_df))
print('Fused payloads    :', len(fused_index_df))
print('Common image_ids  :', len(common_image_ids))

rows = []
for image_id in common_image_ids:
    orig_payload = safe_torch_load(orig_path_lookup[image_id])
    fused_payload = safe_torch_load(fused_path_lookup[image_id])
    for label in TARGET_LABELS:
        try:
            orig_label_key = resolve_label_key(orig_payload, label)
            fused_label_key = resolve_label_key(fused_payload, label)
        except KeyError:
            continue

        orig_item = orig_payload[orig_label_key]
        fused_item = fused_payload[fused_label_key]
        gt_orig = float(orig_item.get('ground_truth', np.nan))
        gt_fused = float(fused_item.get('ground_truth', np.nan))
        gt = gt_orig if np.isfinite(gt_orig) else gt_fused

        rows.append({
            'image_id': str(image_id),
            'label': str(label),
            'orig_label_key': orig_label_key,
            'fused_label_key': fused_label_key,
            'orig_path': Path(orig_path_lookup[image_id]),
            'fused_path': Path(fused_path_lookup[image_id]),
            'ground_truth': gt,
            'orig_probability': float(orig_item.get('positive_probability', np.nan)),
            'orig_margin': float(orig_item.get('margin', np.nan)),
            'fused_probability': float(fused_item.get('positive_probability', np.nan)),
            'fused_margin': float(fused_item.get('margin', np.nan)),
        })

comparison_df = pd.DataFrame(rows)
if comparison_df.empty:
    raise ValueError('No common comparable image-label cases were found.')

comparison_df = comparison_df.sort_values(['label', 'image_id']).reset_index(drop=True)
print('Total comparable cases:', len(comparison_df))
display(comparison_df.groupby('label')[['image_id']].count().rename(columns={'image_id': 'n_cases'}))
display(comparison_df.head(10))

## Optional filtering / deterministic gallery order

In [ ]:
# Optional: uncomment any of the following filters.
# comparison_df = comparison_df[comparison_df['ground_truth'] == 1].reset_index(drop=True)
# comparison_df = comparison_df.head(100).reset_index(drop=True)
# comparison_df = comparison_df[comparison_df['label'].eq('Cardiomegaly')].reset_index(drop=True)

label_order = {label: idx for idx, label in enumerate(TARGET_LABELS)}
gallery_df = comparison_df.assign(
    _label_order=comparison_df['label'].map(lambda x: label_order.get(x, 999))
).sort_values(['_label_order', 'image_id'], kind='stable').drop(columns='_label_order').reset_index(drop=True)

print('Gallery cases:', len(gallery_df))
display(gallery_df.head(10))

## Heatmap access helpers

In [ ]:
def get_original_albef_maps(row):
    payload = safe_torch_load(row.orig_path)
    item = payload[row.orig_label_key]
    vis_up = as_2d_numpy(item['cam_vis_up'])
    vis = as_2d_numpy(item.get('cam_vis', vis_up))
    return {
        'vis_up': vis_up,
        'vis': vis,
        'probability': float(item.get('positive_probability', np.nan)),
        'margin': float(item.get('margin', np.nan)),
        'positive_similarity': float(item.get('positive_similarity', np.nan)),
        'negative_similarity': float(item.get('negative_similarity', np.nan)),
    }


def get_fc_fused_maps(row):
    payload = safe_torch_load(row.fused_path)
    item = payload[row.fused_label_key]
    fused = item['fused']
    return {
        'fused_vis_up': as_2d_numpy(fused['cam_vis_up']),
        'fused_vis': as_2d_numpy(fused.get('cam_vis', fused['cam_vis_up'])),
        'original_branch_vis_up': as_2d_numpy(item['original']['cam_vis_up']),
        'lung_branch_vis_up': as_2d_numpy(item['lung']['cam_vis_up']),
        'heart_branch_vis_up': as_2d_numpy(item['heart']['cam_vis_up']),
        'probability': float(item.get('positive_probability', np.nan)),
        'margin': float(item.get('margin', np.nan)),
        'positive_similarity': float(item.get('positive_similarity', np.nan)),
        'negative_similarity': float(item.get('negative_similarity', np.nan)),
    }

## Inspect one comparison case in detail

In [ ]:
CASE_INDEX = 0  # change this
row = gallery_df.iloc[CASE_INDEX]
image = load_image(row.image_id)
boxes = get_boxes(row.image_id, row.label, image.size)
orig_maps = get_original_albef_maps(row)
fused_maps = get_fc_fused_maps(row)

fig, axes = plt.subplots(1, 7, figsize=(24, 4), dpi=FIG_DPI)
axes[0].imshow(image)
draw_boxes(axes[0], boxes)
axes[0].set_title(f'Original image + GT\n{row.image_id} | {row.label} | y={row.ground_truth}')

axes[1].imshow(orig_maps['vis_up'], cmap=CMAP_NAME, vmin=0, vmax=1)
axes[1].set_title(f'Original ALBEF map\np={orig_maps["probability"]:.4f} | margin={orig_maps["margin"]:.4f}')

axes[2].imshow(make_overlay(image, orig_maps['vis_up']))
draw_boxes(axes[2], boxes)
axes[2].set_title('Original ALBEF overlay')

axes[3].imshow(fused_maps['fused_vis_up'], cmap=CMAP_NAME, vmin=0, vmax=1)
axes[3].set_title(f'FC fused map\np={fused_maps["probability"]:.4f} | margin={fused_maps["margin"]:.4f}')

axes[4].imshow(make_overlay(image, fused_maps['fused_vis_up']))
draw_boxes(axes[4], boxes)
axes[4].set_title('FC fused overlay')

axes[5].imshow(make_overlay(image, fused_maps['lung_branch_vis_up']))
draw_boxes(axes[5], boxes)
axes[5].set_title('FC lung-branch overlay')

axes[6].imshow(make_overlay(image, fused_maps['heart_branch_vis_up']))
draw_boxes(axes[6], boxes)
axes[6].set_title('FC heart-branch overlay')

for ax in axes:
    ax.axis('off')

fig.suptitle(
    f'{CASE_INDEX+1:03d}. Original ALBEF vs FC-fused heatmaps',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.show()

## Render the main comparison PDF

In [ ]:
MAIN_PDF_PATH = VIS_OUTPUT_DIR / 'compare_original_albef_vs_fc_fused_heatmaps.pdf'


def render_main_page(page_df, page_number, start_index):
    n = len(page_df)
    fig, axes = plt.subplots(n, 5, figsize=(18, 3.6 * n), dpi=FIG_DPI, squeeze=False)

    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        boxes = get_boxes(row.image_id, row.label, image.size)
        orig_maps = get_original_albef_maps(row)
        fused_maps = get_fc_fused_maps(row)

        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], boxes)
        axes[r, 0].set_title(
            f'{start_index + r + 1:03d}. {row.image_id}\n{row.label} | GT={row.ground_truth}',
            fontsize=8,
        )

        axes[r, 1].imshow(orig_maps['vis_up'], cmap=CMAP_NAME, vmin=0, vmax=1)
        axes[r, 1].set_title(
            'Original ALBEF map\n'
            f'p={orig_maps["probability"]:.4f} | margin={orig_maps["margin"]:.4f}',
            fontsize=8,
        )

        axes[r, 2].imshow(make_overlay(image, orig_maps['vis_up']))
        draw_boxes(axes[r, 2], boxes)
        axes[r, 2].set_title('Original ALBEF overlay', fontsize=8)

        axes[r, 3].imshow(fused_maps['fused_vis_up'], cmap=CMAP_NAME, vmin=0, vmax=1)
        axes[r, 3].set_title(
            'FC fused map\n'
            f'p={fused_maps["probability"]:.4f} | margin={fused_maps["margin"]:.4f}',
            fontsize=8,
        )

        axes[r, 4].imshow(make_overlay(image, fused_maps['fused_vis_up']))
        draw_boxes(axes[r, 4], boxes)
        axes[r, 4].set_title('FC fused overlay', fontsize=8)

        for ax in axes[r]:
            ax.axis('off')

    fig.suptitle(
        f'Original ALBEF vs FC-fused multi-view heatmaps — page {page_number:02d}',
        fontsize=14,
        fontweight='bold',
        y=1.002,
    )
    plt.tight_layout()
    return fig


num_pages = math.ceil(len(gallery_df) / CASES_PER_PAGE)
print(f'Rendering {len(gallery_df)} cases across {num_pages} pages to {MAIN_PDF_PATH}')

with PdfPages(MAIN_PDF_PATH) as pdf:
    for page_number in range(num_pages):
        start = page_number * CASES_PER_PAGE
        stop = min(len(gallery_df), (page_number + 1) * CASES_PER_PAGE)
        page_df = gallery_df.iloc[start:stop]
        fig = render_main_page(page_df, page_number + 1, start)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print('Saved:', MAIN_PDF_PATH)

## Optional branch-detail PDF

In [ ]:
BRANCH_PDF_PATH = VIS_OUTPUT_DIR / 'compare_fc_fusion_branch_details.pdf'


def render_branch_page(page_df, page_number, start_index):
    n = len(page_df)
    fig, axes = plt.subplots(n, 6, figsize=(22, 3.6 * n), dpi=FIG_DPI, squeeze=False)

    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        boxes = get_boxes(row.image_id, row.label, image.size)
        orig_maps = get_original_albef_maps(row)
        fused_maps = get_fc_fused_maps(row)

        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], boxes)
        axes[r, 0].set_title(
            f'{start_index + r + 1:03d}. {row.image_id}\n{row.label} | GT={row.ground_truth}',
            fontsize=8,
        )

        axes[r, 1].imshow(make_overlay(image, orig_maps['vis_up']))
        draw_boxes(axes[r, 1], boxes)
        axes[r, 1].set_title('Original ALBEF overlay', fontsize=8)

        axes[r, 2].imshow(make_overlay(image, fused_maps['original_branch_vis_up']))
        draw_boxes(axes[r, 2], boxes)
        axes[r, 2].set_title('FC original-branch overlay', fontsize=8)

        axes[r, 3].imshow(make_overlay(image, fused_maps['lung_branch_vis_up']))
        draw_boxes(axes[r, 3], boxes)
        axes[r, 3].set_title('FC lung-branch overlay', fontsize=8)

        axes[r, 4].imshow(make_overlay(image, fused_maps['heart_branch_vis_up']))
        draw_boxes(axes[r, 4], boxes)
        axes[r, 4].set_title('FC heart-branch overlay', fontsize=8)

        axes[r, 5].imshow(make_overlay(image, fused_maps['fused_vis_up']))
        draw_boxes(axes[r, 5], boxes)
        axes[r, 5].set_title('FC final fused overlay', fontsize=8)

        for ax in axes[r]:
            ax.axis('off')

    fig.suptitle(
        f'FC-fusion branch-detail heatmaps — page {page_number:02d}',
        fontsize=14,
        fontweight='bold',
        y=1.002,
    )
    plt.tight_layout()
    return fig


if GENERATE_BRANCH_DETAIL_PDF:
    num_pages = math.ceil(len(gallery_df) / CASES_PER_PAGE)
    print(f'Rendering branch-detail PDF with {len(gallery_df)} cases across {num_pages} pages to {BRANCH_PDF_PATH}')

    with PdfPages(BRANCH_PDF_PATH) as pdf:
        for page_number in range(num_pages):
            start = page_number * CASES_PER_PAGE
            stop = min(len(gallery_df), (page_number + 1) * CASES_PER_PAGE)
            page_df = gallery_df.iloc[start:stop]
            fig = render_branch_page(page_df, page_number + 1, start)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    print('Saved:', BRANCH_PDF_PATH)
else:
    print('GENERATE_BRANCH_DETAIL_PDF=False, so branch-detail PDF was skipped.')

## Save gallery order and diagnostics

In [ ]:
GALLERY_ORDER_CSV = VIS_OUTPUT_DIR / 'compare_original_albef_vs_fc_fusion_gallery_order.csv'
DIAGNOSTICS_CSV = VIS_OUTPUT_DIR / 'compare_original_albef_vs_fc_fusion_diagnostics.csv'
MANIFEST_JSON = VIS_OUTPUT_DIR / 'compare_original_albef_vs_fc_fusion_manifest.json'

gallery_df.to_csv(GALLERY_ORDER_CSV, index=False)
comparison_df.to_csv(DIAGNOSTICS_CSV, index=False)

manifest = {
    'original_albef_dir': str(ORIGINAL_ALBEF_DIR),
    'fused_fc_dir': str(FUSED_FC_DIR),
    'images_root': str(IMAGES_ROOT),
    'annotations_csv': str(ANNOTATIONS_CSV),
    'image_metadata_csv': str(IMAGE_METADATA_CSV),
    'target_labels': TARGET_LABELS,
    'n_cases': int(len(gallery_df)),
    'cases_per_page': int(CASES_PER_PAGE),
    'main_pdf': str(MAIN_PDF_PATH),
    'branch_pdf': str(BRANCH_PDF_PATH) if GENERATE_BRANCH_DETAIL_PDF else None,
    'gallery_order_csv': str(GALLERY_ORDER_CSV),
    'diagnostics_csv': str(DIAGNOSTICS_CSV),
}
with open(MANIFEST_JSON, 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2)

print('Saved:', GALLERY_ORDER_CSV)
print('Saved:', DIAGNOSTICS_CSV)
print('Saved:', MANIFEST_JSON)
print('Saved:', MAIN_PDF_PATH)
if GENERATE_BRANCH_DETAIL_PDF:
    print('Saved:', BRANCH_PDF_PATH)

## Notes

- The **main PDF** compares the original single-view ALBEF heatmap directly against the **final fused FC heatmap**.
- The **branch-detail PDF** is optional and shows how the original/lung/heart branches contributed inside the FC-fused model.
- The overlay uses each map's own normalized `cam_vis_up`, so visual intensity is comparable **within** a case but not as an absolute score across different images.
- GT boxes are drawn only when the requested image-label pair has a VinDr bounding-box annotation.